# أثر تحسين العتبة (Effect of Threshold Optimization) — D-MorphNet

هذا الدفتر ينفّذ قسم **"Effect of Threshold Optimization"**: اختيار عتبة التصنيف الصحيحة أمر جوهري لأداء نظام كشف المورف.

## لماذا العتبة مهمة؟

قرار النظام يعتمد على مقارنة درجة القرار $s = w^TF + b$ بعتبة $\tau$:

$$\hat{y} = \begin{cases} +1\ \text{(مدموجة)} & s \ge \tau \\ -1\ \text{(حقيقية)} & s < \tau \end{cases}$$

- العتبة الافتراضية هي $\tau = 0$ (الحد الفاصل نفسه)، لكنها ليست بالضرورة الأمثل.
- عتبة **منخفضة جداً** ← يصنّف صوراً حقيقية كثيرة كمورف خطأً (FP يرتفع).
- عتبة **مرتفعة جداً** ← تفوته صور مورف (FN يرتفع — الأخطر أمنياً).

## منهجية الورقة (التي سننفّذها)
1. تجربة قيم عتبة مختلفة ومراقبة تغيّر تنبؤات النظام للصور الحقيقية والمدموجة.
2. **الاختيار يتم على مجموعة التحقق فقط** — لا نلمس الاختبار أثناء الضبط حتى يبقى التقييم عادلاً.
3. اختيار القيمة التي تحقق أفضل توازن بين اكتشاف المورف وتجنّب تصنيف الحقيقي خطأً.
4. تطبيق العتبة المختارة على مجموعة الاختبار — في الورقة حقق النظام **89.9%** بالعتبة المختارة.


## ١) تجهيز البيئة


In [ ]:
!pip -q install scikit-learn joblib pandas

import os, glob, random, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PAPER_TEST_ACC = 0.899     # الدقة المنشورة على الاختبار بالعتبة المختارة
print("✅ البيئة جاهزة")


## ٢) تحميل السمات والمصنّف وحساب درجات القرار

نحمّل سمات **التحقق** (لاختيار العتبة) و**الاختبار** (للتقييم النهائي فقط) ومصنّف SVM النهائي، ثم نحسب $s = w^TF+b$ لكل صورة في المجموعتين.


In [ ]:
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

FEAT_DIR = "/content/DMorphNet_features"
DRIVE_DIR = "/content/drive/MyDrive/DMorphNet_models"
SPLITS = ["train", "val", "test"]


def find_file(name):
    for root in [FEAT_DIR, DRIVE_DIR]:
        p = os.path.join(root, name)
        if os.path.exists(p):
            return p
    return None


if find_file("effb6_ft_train.npz") is None and find_file("effb6_train.npz") is None:
    from google.colab import drive
    drive.mount('/content/drive')

PREFIX = "effb6_ft" if find_file("effb6_ft_train.npz") else "effb6"
F, y = {}, {}
for split in SPLITS:
    data = np.load(find_file(f"{PREFIX}_{split}.npz"))
    F[split] = data["X"].astype(np.float32)
    y[split] = np.where(data["y"] == 0, -1, +1)      # -1 حقيقية ، +1 مدموجة

svm_path, sc_path = find_file("svm_hybrid_final.joblib"), find_file("scaler_hybrid_final.joblib")
if svm_path and sc_path:
    svm_final, scaler = joblib.load(svm_path), joblib.load(sc_path)
    print("تم تحميل المصنّف النهائي المحفوظ")
else:
    print("لا يوجد مصنّف محفوظ — إعادة تدريب سريعة ...")
    scaler = StandardScaler().fit(F["train"])
    svm_final = LinearSVC(C=1.0, random_state=SEED)
    svm_final.fit(scaler.transform(F["train"]), y["train"])

scores = {s: svm_final.decision_function(scaler.transform(F[s]))
          for s in ["val", "test"]}
print(f"درجات التحقق: {len(scores['val'])} — درجات الاختبار: {len(scores['test'])}")


## ٣) مسح قيم العتبة على مجموعة التحقق

نجرّب شبكة كثيفة من قيم $\tau$ عبر كامل مدى درجات التحقق، ونحسب عند كل قيمة: الدقة، وF1، ومعدل اكتشاف المورف (TPR)، ومعدل الإنذار الخاطئ (FPR) — لنرى بالضبط كيف تتغير تنبؤات النظام للصور الحقيقية والمدموجة مع تحرك العتبة.


In [ ]:
from sklearn.metrics import accuracy_score, f1_score

s_val, y_val = scores["val"], y["val"]

taus = np.linspace(s_val.min(), s_val.max(), 400)
sweep = []
for tau in taus:
    pred = np.where(s_val >= tau, +1, -1)
    tp = int(((pred == +1) & (y_val == +1)).sum())
    fn = int(((pred == -1) & (y_val == +1)).sum())
    fp = int(((pred == +1) & (y_val == -1)).sum())
    tn = int(((pred == -1) & (y_val == -1)).sum())
    sweep.append({
        "tau": tau,
        "accuracy": (tp + tn) / len(y_val),
        "f1": f1_score(y_val, pred, pos_label=+1, zero_division=0),
        "tpr": tp / max(tp + fn, 1),      # اكتشاف المورف
        "fpr": fp / max(fp + tn, 1),      # إنذار خاطئ على الحقيقي
    })

sweep = pd.DataFrame(sweep)
print(f"تم فحص {len(sweep)} قيمة عتبة على مجموعة التحقق")
sweep.head()


## ٤) اختيار العتبة المتوازنة

نرسم المقاييس مقابل قيمة العتبة ونختار $\tau^{*}$ التي تعظّم **F1 على التحقق** — وهو المقياس الذي يوازن بين اكتشاف المورف (Recall) وتجنّب اتهام الصور الحقيقية خطأً (Precision)، تماماً كوصف الورقة: "توازن جيد بين اكتشاف صور المورف وتجنّب أخطاء الصور الحقيقية".


In [ ]:
best_idx = int(sweep["f1"].idxmax())
TAU_STAR = float(sweep.loc[best_idx, "tau"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# المقاييس مقابل العتبة
axes[0].plot(sweep["tau"], sweep["accuracy"], label="الدقة (Accuracy)")
axes[0].plot(sweep["tau"], sweep["f1"], label="F1")
axes[0].axvline(TAU_STAR, color="green", linestyle="--",
                label=f"العتبة المختارة τ* = {TAU_STAR:.3f}")
axes[0].axvline(0, color="gray", linestyle=":", label="الافتراضية τ = 0")
axes[0].set_xlabel("قيمة العتبة τ")
axes[0].set_ylabel("قيمة المقياس")
axes[0].set_title("الدقة و F1 مقابل العتبة (تحقق)")
axes[0].legend(); axes[0].grid(alpha=0.3)

# سلوك الفئتين مقابل العتبة
axes[1].plot(sweep["tau"], sweep["tpr"], label="TPR: اكتشاف المورف")
axes[1].plot(sweep["tau"], sweep["fpr"], label="FPR: إنذار خاطئ على الحقيقي")
axes[1].axvline(TAU_STAR, color="green", linestyle="--",
                label=f"τ* = {TAU_STAR:.3f}")
axes[1].set_xlabel("قيمة العتبة τ")
axes[1].set_ylabel("المعدل")
axes[1].set_title("أثر العتبة على الفئتين (تحقق)")
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

row = sweep.loc[best_idx]
print(f"العتبة المختارة على التحقق: τ* = {TAU_STAR:.4f}")
print(f"  عند τ*: دقة التحقق = {row['accuracy']:.4f} ، F1 = {row['f1']:.4f} ، "
      f"TPR = {row['tpr']:.4f} ، FPR = {row['fpr']:.4f}")


## ٥) التقييم النهائي على الاختبار: الافتراضية مقابل المختارة

نطبّق العتبتين على مجموعة **الاختبار** (لأول مرة) ونقارن كل المقاييس — في الورقة بلغت الدقة **89.9%** بالعتبة المختارة.


In [ ]:
from sklearn.metrics import precision_score, recall_score, confusion_matrix

s_test, y_test = scores["test"], y["test"]


def evaluate_at(tau):
    pred = np.where(s_test >= tau, +1, -1)
    cm = confusion_matrix(y_test, pred, labels=[-1, +1])
    return pred, {
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, pos_label=+1),
        "Recall": recall_score(y_test, pred, pos_label=+1),
        "F1": f1_score(y_test, pred, pos_label=+1),
        "TN": int(cm[0, 0]), "FP": int(cm[0, 1]),
        "FN": int(cm[1, 0]), "TP": int(cm[1, 1]),
    }


pred_def, m_def = evaluate_at(0.0)
pred_opt, m_opt = evaluate_at(TAU_STAR)

table = pd.DataFrame({
    "العتبة الافتراضية (τ=0)": m_def,
    f"العتبة المختارة (τ*={TAU_STAR:.3f})": m_opt,
}).T
print("نتائج مجموعة الاختبار:")
display(table.round(4))

print(f"دقة الاختبار بالعتبة المختارة: {m_opt['Accuracy']*100:.2f}%  "
      f"(الورقة: {PAPER_TEST_ACC*100:.1f}%)")
delta = (m_opt["Accuracy"] - m_def["Accuracy"]) * 100
print(f"التغيّر في الدقة بعد تحسين العتبة: {delta:+.2f} نقطة مئوية")
print(f"التغيّر في FN (المورف الفائت — الأخطر): {m_def['FN']} ← {m_opt['FN']}")


## ٦) مصفوفتا الالتباس جنباً إلى جنب

مقارنة بصرية: كيف أعادت العتبة المختارة توزيع الأخطاء بين FP (حقيقي متهم خطأً) وFN (مورف فائت).


In [ ]:
def plot_cm(ax, tau, pred, title):
    cm = confusion_matrix(y_test, pred, labels=[-1, +1])
    ax.imshow(cm, cmap="Blues")
    ax.set_title(title)
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(["Real", "Morph"]); ax.set_yticklabels(["Real", "Morph"])
    ax.set_xlabel("Predicted label"); ax.set_ylabel("True label")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=13,
                    color="white" if cm[i, j] > cm.max() / 2 else "black")


fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plot_cm(axes[0], 0.0, pred_def, "العتبة الافتراضية τ = 0")
plot_cm(axes[1], TAU_STAR, pred_opt, f"العتبة المختارة τ* = {TAU_STAR:.3f}")
plt.tight_layout()
plt.show()


## ٧) توزيع درجات القرار مع العتبتين

نرى مكان العتبتين وسط توزيعي الفئتين: المنطقة المتداخلة بين التوزيعين هي مصدر كل الأخطاء، وموضع العتبة يحدد كيف تُقسَّم تلك المنطقة بين FP وFN.


In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(s_test[y_test == -1], bins=60, alpha=0.6, label="حقيقية (y = -1)")
plt.hist(s_test[y_test == +1], bins=60, alpha=0.6, label="مدموجة (y = +1)")
plt.axvline(0, color="gray", linestyle=":", linewidth=2,
            label="الافتراضية τ = 0")
plt.axvline(TAU_STAR, color="green", linestyle="--", linewidth=2,
            label=f"المختارة τ* = {TAU_STAR:.3f}")
plt.xlabel("درجة القرار  s = wᵀF + b")
plt.ylabel("عدد الصور")
plt.title("توزيع درجات القرار على الاختبار وموضع العتبتين")
plt.legend()
plt.tight_layout()
plt.show()

print("الخلاصة: لتحسين موثوقية النظام يجب اعتماد العتبة المضبوطة على التحقق،")
print("لا العتبة الافتراضية — فهي التي تحقق التوازن الصحيح بين FP و FN.")


## ٨) حفظ العتبة المختارة والنتائج في Google Drive

نحفظ قيمة العتبة المختارة مع نتائج المقارنة — أي نظام نشر لاحق يجب أن يقرأ هذه العتبة ويستخدمها بدل الافتراضية.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

RESULTS_DIR = "/content/drive/MyDrive/DMorphNet_results"
MODELS_DIR = "/content/drive/MyDrive/DMorphNet_models"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

np.savez(os.path.join(MODELS_DIR, "optimal_threshold.npz"),
         tau=TAU_STAR, criterion="max F1 on validation")
table.to_csv(os.path.join(RESULTS_DIR, "threshold_optimization_results.csv"))
sweep.to_csv(os.path.join(RESULTS_DIR, "threshold_sweep_validation.csv"),
             index=False)

print("تم حفظ العتبة المختارة في:",
      os.path.join(MODELS_DIR, "optimal_threshold.npz"))
print("تم حفظ النتائج في:", RESULTS_DIR)
print("\n🎉 اكتمل قسم أثر تحسين العتبة")
